# distribution shift — Python demo

Numerical companion to the entry [distribution shift](https://dictionaryofml.org/terms/distshift.html) of the [Dictionary of Applied Machine Learning](https://dictionaryofml.org/): it recomputes what the entry states and prints one line per check.

Numerical companion to the glossary entry 'distshift' (distribution shift). The GeoSphere Austria station Krems an der Donau (3805) records the air temperature every hour. This script downloads the hourly record for December 2025 - February 2026 and for June-August 2026, keeps the days carrying both the 06:00 and the 15:00 reading (UTC), and writes them to CSV files, so the exact numbers behind the figure stay on record. Checks pin the downloaded values to the archive, so a change on the server side is caught rather than silently absorbed.

Requires NumPy and Matplotlib only, and uses fixed seeds, so the printed numbers reproduce exactly. Generated from [`pythondemos/distshift.py`](https://dictionaryofml.org/terms/distshift.py); CC BY 4.0.

Before running a downloaded copy, scan it: `uvx bandit -q -t B102,B307,B602,B605 distshift.py` (B102,B307,B602,B605 are the exec, eval and shell checks of [bandit](https://bandit.readthedocs.io/)) reports nothing: the script runs no shell command and evaluates no generated code.

In [ ]:
# Notebook shim: the script resolves output paths relative to __file__,
# which a notebook kernel does not define; everything lands in the
# working directory instead.
import os
__file__ = os.path.join(os.getcwd(), "distshift.py")
os.makedirs("pythondemos", exist_ok=True)

In [ ]:
"""Distribution shift between two seasons at Krems an der Donau: a
linear hypothesis learned from winter days fails on summer days.

Purpose
-------
Numerical companion to the glossary entry 'distshift' (distribution
shift).  The GeoSphere Austria station Krems an der Donau (3805)
records the air temperature every hour.  This script downloads the
hourly record for December 2025 - February 2026 and for June-August
2026, keeps the days carrying both the 06:00 and the 15:00 reading
(UTC), and writes them to CSV files, so the exact numbers behind the
figure stay on record.  Checks pin the downloaded values to the
archive, so a change on the server side is caught rather than silently
absorbed.

Each data point is one day: its feature is the 06:00 reading of that
day, its label the 15:00 reading of the same day.  A linear hypothesis
is learned from winter days by minimizing the average squared error on
the training set; its average squared error on a held-out test set from
the same season is close to the training error.  Applied to the summer
days, the same hypothesis errs 37 times worse: every summer feature
lies above every winter feature (the distribution of the feature values
has shifted), and the relation between the two readings differs as well
(the conditional distribution has shifted too), so the winter line
undershoots every summer afternoon, by 11.4 degrees on average.

Deterministic: the only randomness is the train/test split of the
winter days, drawn with a fixed seed.  Self-contained: numpy +
matplotlib only (stdlib urllib for the download).

Blocks
------
[B-fetch] Download the hourly record of December 2025 - February 2026
          and of June-August 2026 at Krems, keep the days carrying both
          the 06:00 and the 15:00 reading, write them to
          distshift_winter.csv and distshift_summer.csv and check them
          against the archive.
[B-split] Split the winter days into a training set and a test set
          with a fixed seed; check the sizes.
[B-learn] Learn the linear hypothesis by minimizing the average squared
          error on the training set; check the learned line.
[B-shift] Average squared error on the training set, the test set, and
          the summer days; check that the test error stays close to the
          training error while the summer error is 37 times larger, that
          the two feature ranges do not overlap, that the hypothesis
          UNDERestimates every summer label, and that a line fitted to
          the summer days has a different slope (the conditional
          distribution shifted too).
[B-cov]   Covariate shift, in contrast to the two-season example.
          Five calendar years at Krems (2021-2025) form a pool of 1823
          days that carry both readings.  Two samples are drawn from it
          by accepting each day with a probability that depends on its
          FEATURE alone: the training sample favours cold mornings, the
          deployment sample warm ones.  The label is never consulted, so
          the conditional distribution of the label given the feature is
          the same in both samples by construction.  The checks confirm
          this where it can be seen without a model, by comparing the
          mean label in matched feature bins, and record that the best
          straight line nevertheless differs between the two samples --
          a straight line cannot represent the conditional exactly, so
          the line it fits depends on where the features sit.

Outputs
-------
distshift_winter.csv     : date, t06, t15 of the winter days
distshift_summer.csv     : date, t06, t15 of the summer days
distshift_train.csv      : t06, t15 of the winter training set
distshift_test.csv       : t06, t15 of the winter test set
distshift_deploy.csv     : t06, t15 of the summer days
distshift_*_plot.csv     : every 6th / 4th / 5th of those, the markers of the figure
distshift_hypothesis.csv : t06, t15 along the learned hypothesis
distshift_errors.csv     : average squared error on the three sets
distshift.png            : preview (checking only) -- the data points,
                           the learned hypothesis, and the three errors
distshift_cov_hist_*.csv : feature histogram of the training and the
                           deployment sample of [B-cov]
distshift_cov_cond.csv   : mean label per matched feature bin, both
                           samples of [B-cov]
"""

# /// script
# requires-python = ">=3.10"
# dependencies = ["matplotlib==3.10.7", "numpy==2.3.4"]
# ///

import json
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = Path(__file__).parent

report = []                         # collects (check name, pass/fail) pairs


def check(name, ok):                # records and prints one verification
    report.append((name, bool(ok)))
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")

**[B-fetch]** Download the hourly record of December 2025 - February 2026 and of June-August 2026 at Krems, keep the days carrying both the 06:00 and the 15:00 reading, write them to distshift_winter.csv and distshift_summer.csv and check them against the archive.

In [ ]:
def fetch_days(start, end):
    """Daily (date, t06, t15) records at Krems, days with a gap dropped.

    The hourly archive stamps are UTC and so are the two times, so the
    filter is the stamp itself; whole days are missing here and there,
    and a day is kept only when it carries both readings."""
    url = ("https://dataset.api.hub.geosphere.at/v1/station/historical/"
           "klima-v2-1h?parameters=tl&station_ids=3805"
           f"&start={start}T00:00&end={end}T23:00")
    with urllib.request.urlopen(url, timeout=300) as resp:
        payload = json.load(resp)
    readings = payload["features"][0]["properties"]["parameters"]["tl"]["data"]
    values = {}                     # date -> {"06:00": v, "15:00": v}
    for stamp, value in zip(payload["timestamps"], readings):
        if value is None:
            continue
        hour = stamp[11:16]
        if hour in ("06:00", "15:00"):
            values.setdefault(stamp[:10], {})[hour] = float(value)
    return [(day, rec["06:00"], rec["15:00"])
            for day, rec in sorted(values.items())
            if len(rec) == 2
            and np.isfinite([rec["06:00"], rec["15:00"]]).all()]


summer = fetch_days("2026-06-01", "2026-08-31")
winter = fetch_days("2025-12-01", "2026-02-28")
for fname, records in (("distshift_summer.csv", summer),
                       ("distshift_winter.csv", winter)):
    with open(OUT_DIR / fname, "w") as f:
        f.write("date,t06,t15\n")
        for day, lo, hi in records:
            f.write(f"{day},{lo},{hi}\n")
check("[B-fetch] 92 summer days downloaded", len(summer) == 92)
check("[B-fetch] 88 winter days downloaded", len(winter) == 88)
check(f"[B-fetch] the summer record matches the archive "
      f"(Jun 1: {summer[0][1]} at 06:00, {summer[0][2]} at 15:00)",
      summer[0][0] == "2026-06-01")
check(f"[B-fetch] the winter record matches the archive "
      f"(Dec 1: {winter[0][1]} at 06:00, {winter[0][2]} at 15:00)",
      winter[0][0] == "2025-12-01")

**[B-split]** Split the winter days into a training set and a test set with a fixed seed; check the sizes.

In [ ]:
temps_summer = np.array([[lo, hi] for _, lo, hi in summer])
temps_winter = np.array([[lo, hi] for _, lo, hi in winter])
rng = np.random.default_rng(42)
order = rng.permutation(len(temps_winter))
train, test = temps_winter[order[:59]], temps_winter[order[59:]]
check("[B-split] 59 training and 29 test days", len(train) == 59
      and len(test) == 29)

**[B-learn]** Learn the linear hypothesis by minimizing the average squared error on the training set; check the learned line.

In [ ]:
A = np.stack([train[:, 0], np.ones(len(train))], 1)
w1, w0 = np.linalg.lstsq(A, train[:, 1], rcond=None)[0]
print(f"  learned hypothesis: t15 = {w1:.3f} * t06 + {w0:.3f}")
check(f"[B-learn] the learned line is t15 = {w1:.3f} * t06 + {w0:.3f}",
      abs(w1 - 0.642) < 5e-3 and abs(w0 - 3.390) < 5e-3)


def avg_sq_err(points):
    """Average squared error of the learned hypothesis on (t06, t15)."""
    return float(np.mean((w1 * points[:, 0] + w0 - points[:, 1]) ** 2))

**[B-shift]** Average squared error on the training set, the test set, and the summer days; check that the test error stays close to the training error while the summer error is 37 times larger, that the two feature ranges do not overlap, that the hypothesis UNDERestimates every summer label, and that a line fitted to the summer days has a different slope (the conditional distribution shifted too).

In [ ]:
err_train, err_test = avg_sq_err(train), avg_sq_err(test)
err_deploy = avg_sq_err(temps_summer)
print(f"  avg squared error: train {err_train:.2f}, test {err_test:.2f},"
      f" summer after deployment {err_deploy:.2f}")
check("[B-shift] the test error is within a factor 2 of the training error",
      err_test < 2 * err_train)
check(f"[B-shift] the deployment error exceeds ten times the test error "
      f"({err_deploy:.1f} against {err_test:.1f}, a factor "
      f"{err_deploy / err_test:.0f})", err_deploy > 10 * err_test)
check("[B-shift] the two feature ranges do not overlap: every summer "
      "morning is warmer than every winter morning",
      temps_winter[:, 0].max() < temps_summer[:, 0].min())
errs_deploy = w1 * temps_summer[:, 0] + w0 - temps_summer[:, 1]
check(f"[B-shift] every summer label is UNDERestimated, by "
      f"{-errs_deploy.mean():.1f} degrees on average",
      bool((errs_deploy < 0).all()))
check(f"[B-shift] the mean summer day reads "
      f"{temps_summer[:, 0].mean():.1f} at 06:00 and "
      f"{temps_summer[:, 1].mean():.1f} at 15:00",
      temps_summer[:, 1].mean() > temps_summer[:, 0].mean())
w1_sum, w0_sum = np.linalg.lstsq(
    np.stack([temps_summer[:, 0], np.ones(len(temps_summer))], 1),
    temps_summer[:, 1], rcond=None)[0]
print(f"  line fitted to the summer days: t15 = {w1_sum:.3f} * t06"
      f" + {w0_sum:.3f}")
check(f"[B-shift] the summer days follow a different line (slope "
      f"{w1_sum:.3f} against {w1:.3f})", abs(w1_sum - w1) > 0.3)

**[B-cov]** Covariate shift, in contrast to the two-season example. Five calendar years at Krems (2021-2025) form a pool of 1823 days that carry both readings. Two samples are drawn from it by accepting each day with a probability that depends on its FEATURE alone: the training sample favours cold mornings, the deployment sample warm ones. The label is never consulted, so the conditional distribution of the label given the feature is the same in both samples by construction. The checks confirm this where it can be seen without a model, by comparing the mean label in matched feature bins, and record that the best straight line nevertheless differs between the two samples -- a straight line cannot represent the conditional exactly, so the line it fits depends on where the features sit.

In [ ]:
#   The two-station example above is NOT covariate shift: its conditional
#   distribution moves as well.  This block constructs a shift that is
#   covariate shift by construction, from a single pool of days, by
#   letting the acceptance probability depend on the 06:00 reading alone.
pool = []
for year in range(2021, 2026):     # one call per year: the API rejects a
    pool += fetch_days(f"{year}-01-01", f"{year}-12-31")
check(f"[B-cov] {len(pool)} days of Krems 2021-2025 carry both readings",
      len(pool) > 1800)
temps_pool = np.array([[lo, hi] for _, lo, hi in pool])
feat, lab = temps_pool[:, 0], temps_pool[:, 1]

rng_cov = np.random.default_rng(7)     # separate from the [B-split] seed
CENTRE, WIDTH = 4.0, 4.0               # logistic acceptance, in deg C
accept_train = 1.0 / (1.0 + np.exp((feat - CENTRE) / WIDTH))
accept_deploy = 1.0 / (1.0 + np.exp(-(feat - CENTRE) / WIDTH))
sel_train = rng_cov.random(len(feat)) < accept_train
sel_deploy = rng_cov.random(len(feat)) < accept_deploy
check(f"[B-cov] {int(sel_train.sum())} training and "
      f"{int(sel_deploy.sum())} deployment days selected",
      sel_train.sum() > 500 and sel_deploy.sum() > 500)
check(f"[B-cov] the feature mean moves from "
      f"{feat[sel_train].mean():+.1f} to {feat[sel_deploy].mean():+.1f} deg C",
      feat[sel_deploy].mean() - feat[sel_train].mean() > 5.0)

# the conditional, where it can be compared without fitting anything:
# the mean label among the days whose feature falls in the same bin
EDGES = np.arange(-8, 21, 4)
cond_rows = []
for lo, hi in zip(EDGES[:-1], EDGES[1:]):
    in_bin = (feat >= lo) & (feat < hi)
    a, b = sel_train & in_bin, sel_deploy & in_bin
    if a.sum() >= 25 and b.sum() >= 25:
        cond_rows.append((lo + 2.0, int(a.sum()), float(lab[a].mean()),
                          int(b.sum()), float(lab[b].mean())))
check("[B-cov] four feature bins hold at least 25 days of each sample",
      len(cond_rows) == 4)
gap = max(abs(r[2] - r[4]) for r in cond_rows)
check("[B-cov] the mean label per bin agrees within 0.6 deg C "
      f"(largest gap {gap:.2f})", gap < 0.6)


def line_on(mask):
    """Least-squares line through the selected days."""
    design = np.stack([feat[mask], np.ones(int(mask.sum()))], 1)
    return np.linalg.lstsq(design, lab[mask], rcond=None)[0]


cov_w1, cov_w0 = line_on(sel_train)
dep_w1, _ = line_on(sel_deploy)
check(f"[B-cov] the conditional is unchanged by construction, yet the "
      f"best straight line still differs ({cov_w1:.2f} against "
      f"{dep_w1:.2f}): least squares weights the feature range the sample "
      f"covers", abs(cov_w1 - dep_w1) < 0.3)
err_cov_train = float(np.mean((cov_w1 * feat[sel_train] + cov_w0
                               - lab[sel_train]) ** 2))
err_cov_deploy = float(np.mean((cov_w1 * feat[sel_deploy] + cov_w0
                                - lab[sel_deploy]) ** 2))
check(f"[B-cov] the error barely moves ({err_cov_train:.1f} on the "
      f"training sample, {err_cov_deploy:.1f} after deployment)",
      err_cov_deploy < 1.5 * err_cov_train)

HIST = np.arange(-24, 25, 3)
for name, mask in (("distshift_cov_hist_train.csv", sel_train),
                   ("distshift_cov_hist_deploy.csv", sel_deploy)):
    counts, _ = np.histogram(feat[mask], bins=HIST)
    share = counts / counts.sum()
    with open(OUT_DIR / name, "w") as f:
        f.write("t06,share\n")
        for centre, v in zip(HIST[:-1] + 1.5, share):
            f.write(f"{centre:.1f},{v:.4f}\n")
with open(OUT_DIR / "distshift_cov_cond.csv", "w") as f:
    f.write("t06,n_train,mean_train,n_deploy,mean_deploy\n")
    for centre, na, ma, nb, mb in cond_rows:
        f.write(f"{centre:.1f},{na},{ma:.2f},{nb},{mb:.2f}\n")

# ---- outputs: CSVs for the entry's pgfplots figure, preview PNG
header = "t06,t15"
np.savetxt(OUT_DIR / "distshift_train.csv", train, delimiter=",",
           header=header, comments="", fmt="%.1f")
np.savetxt(OUT_DIR / "distshift_test.csv", test, delimiter=",",
           header=header, comments="", fmt="%.1f")
np.savetxt(OUT_DIR / "distshift_deploy.csv", temps_winter, delimiter=",",
           header=header, comments="", fmt="%.1f")
# the entry's figure draws every 6th training day, every 4th test day and
# every 5th deployment day (11, 8 and 18 markers), so the three sets stay
# apart on the page; every number of the entry uses the full sets
for name, arr, step in [("distshift_train_plot.csv", train, 6),
                        ("distshift_test_plot.csv", test, 4),
                        ("distshift_deploy_plot.csv", temps_winter, 5)]:
    np.savetxt(OUT_DIR / name, arr[::step], delimiter=",",
               header=header, comments="", fmt="%.1f")
check("[B-figure] the plotted subsets hold at most 21 days each",
      max(train[::6].shape[0], test[::4].shape[0], temps_winter[::5].shape[0]) <= 21)
t_line = np.array([temps_winter[:, 0].min() - 1.0, temps_summer[:, 0].max() + 1.0])
np.savetxt(OUT_DIR / "distshift_hypothesis.csv",
           np.stack([t_line, w1 * t_line + w0], 1), delimiter=",",
           header=header, comments="", fmt="%.3f")
with open(OUT_DIR / "distshift_errors.csv", "w") as f:
    f.write("set,avg_sq_err\n")
    f.write(f"train,{err_train:.2f}\ntest,{err_test:.2f}\n")
    f.write(f"deploy,{err_deploy:.2f}\n")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(train[:, 0], train[:, 1], s=22, c="black", marker="o",
           label="training set (winter)")
ax.scatter(test[:, 0], test[:, 1], s=26, facecolors="none",
           edgecolors="black", marker="s",
           label="test set (winter)")
ax.scatter(temps_winter[:, 0], temps_winter[:, 1], s=26, facecolors="none",
           edgecolors="black", marker="D",
           label="after deployment (summer)")
ax.plot(t_line, w1 * t_line + w0, "k-", lw=1.5,
        label="learned hypothesis")
ax.set_xlabel("daily minimum temperature (deg C)")
ax.set_ylabel("daily maximum temperature (deg C)")
ax.set_title("Learned from winter days at Krems, deployed on summer days\n"
             f"avg squared error: train {err_train:.1f}, test {err_test:.1f},"
             f" deployment {err_deploy:.1f}")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig(OUT_DIR / "distshift.png", dpi=120)

failed = [name for name, ok in report if not ok]
print(f"{len(report) - len(failed)}/{len(report)} checks passed"
      + (f"; FAILED: {failed}" if failed else ""))
raise SystemExit(1 if failed else 0)